# FLOWER noise-scale mismatch benchmark on Kaggle

This notebook runs the complete project protocol on a Kaggle GPU: 20 fixed CelebA images, 20 fixed AFHQ-Cat images, five inverse problems, and five assumed noise ratios. It produces 1,000 reconstructions and evaluates PSNR, SSIM, and LPIPS.

Before starting, open the notebook settings, select **GPU** as the accelerator, and enable **Internet**. For a background run that is independent of the browser, use **Save Version > Save & Run All**.

## 1. Verify the Kaggle GPU

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Notebook options, set Accelerator to GPU and start a new session.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)

## 2. Clone the clean project repository

In [ ]:
repo_url = 'https://github.com/ItayShorian07/Diffusion--flower-project.git'
repo_dir = Path('/kaggle/working/Diffusion--flower-project')
if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository:', repo_dir)

## 3. Install the small set of additional packages and prepare the official inputs

The scripts download the two official FLOWER checkpoints and the official dataset archives, but extract only the fixed 20-image samples used by this project.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['bash', 'scripts/download_benchmark_models.sh'], check=True)
subprocess.run(['bash', 'scripts/prepare_benchmark_data.sh'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## 4. Configure the Kaggle output directory

Kaggle preserves files under `/kaggle/working` as notebook-version outputs after a successful **Save & Run All** execution.

In [ ]:
output_root = Path('/kaggle/working/RobustFLOWER')
results_dir = output_root / 'main'
figures_dir = output_root / 'figures'
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
print('Results:', results_dir)
print('Figures:', figures_dir)

## 5. Run the complete benchmark

The five assumed noise ratios are processed together for each image and task. The runner writes `metrics.csv` after every completed image-task group. The complete run contains 1,000 result rows.

In [ ]:
command = [
    sys.executable, 'experiments/run_noise_mismatch.py',
    '--device', 'cuda',
    '--image-batch-size', '1',
    '--output', str(results_dir),
]
subprocess.run(command, check=True)

## 6. Aggregate results and create the five predefined figures

In [ ]:
subprocess.run([sys.executable, 'experiments/summarize_results.py', '--input', str(results_dir / 'metrics.csv'), '--output', str(results_dir)], check=True)
subprocess.run([sys.executable, 'experiments/capture_analysis.py', '--device', 'cuda', '--input', str(results_dir)], check=True)
subprocess.run([sys.executable, 'experiments/plot_results.py', '--input', str(results_dir), '--output', str(figures_dir)], check=True)

## 7. Validate and package the deliverable

After the version finishes, open the right-hand **Output** panel and download `robust_flower_results.zip`. Send that ZIP back for final report and presentation generation.

In [ ]:
metrics_path = results_dir / 'metrics.csv'
completed_rows = sum(1 for _ in metrics_path.open()) - 1
if completed_rows != 1000:
    raise RuntimeError(f'Expected 1000 completed rows, found {completed_rows}')

figure_names = sorted(path.name for path in figures_dir.glob('*.png'))
if len(figure_names) != 5:
    raise RuntimeError(f'Expected five figures, found {figure_names}')

archive_path = Path('/kaggle/working/robust_flower_results.zip')
if archive_path.exists():
    archive_path.unlink()
archive = shutil.make_archive(str(archive_path.with_suffix('')), 'zip', output_root)
print('Completed rows:', completed_rows)
print('Figures:', figure_names)
print('Download from Notebook Output:', archive)